# Transformer 英译中实验

本 notebook 用于在 Linux 服务器上启动、评估和恢复 Transformer 训练。运行前请先按 `DEPLOYMENT.md` 创建并激活 `.venv`，然后从该虚拟环境启动 JupyterLab。

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'scripts' / 'train.py').exists(), '请在项目根目录打开此 notebook'
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 设置实验参数

先用较小模型确认环境，再逐步提高 `epochs`、`d_model` 和层数。GPU 显存不足时优先降低 `batch_size`。

In [ ]:
EXPERIMENT = 'baseline-256'
EPOCHS = 50
BATCH_SIZE = 128
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 4
D_FF = 1024
MAX_EXAMPLES = None  # 设为 2000 可先做快速试跑
CHECKPOINT = PROJECT_ROOT / 'checkpoints' / f'{EXPERIMENT}.pt'
CHECKPOINT.parent.mkdir(exist_ok=True)
CHECKPOINT

In [ ]:
import subprocess

command = [
    sys.executable, 'scripts/train.py',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--d-model', str(D_MODEL),
    '--num-heads', str(NUM_HEADS),
    '--num-layers', str(NUM_LAYERS),
    '--d-ff', str(D_FF),
    '--checkpoint-path', str(CHECKPOINT),
]
if MAX_EXAMPLES is not None:
    command += ['--max-examples', str(MAX_EXAMPLES)]
print(' '.join(command))
subprocess.run(command, check=True)

## 评估与翻译

训练完成后运行下面两格。初期输出可能不通顺；这通常需要更长训练、更大语料和更合适的分词策略来改善。

In [ ]:
subprocess.run([sys.executable, 'scripts/evaluate.py', str(CHECKPOINT)], check=True)

In [ ]:
sentence = 'How are you?'
subprocess.run([sys.executable, 'scripts/translate.py', str(CHECKPOINT), sentence], check=True)

## 恢复训练

把 `EPOCHS` 设为需要追加的 epoch 数，然后运行下方单元。新的 checkpoint 会写到 `resumed-...pt`，原文件保留。

In [ ]:
ADDITIONAL_EPOCHS = 20
RESUMED_CHECKPOINT = CHECKPOINT.with_name(f'resumed-{CHECKPOINT.name}')
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--resume', str(CHECKPOINT),
    '--epochs', str(ADDITIONAL_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--checkpoint-path', str(RESUMED_CHECKPOINT),
], check=True)